## Exercise 05.1 (random numbers)

- Using the `randint` function from the `random` module (https://docs.python.org/3/library/random.html#random.randint) to
  develop a function `dice_roll` that emulates the roll of a dice with $n$ sides. The number of sides `n` should an argument to the function.

- For $n=6$, devise and implement a test to check that it is a fair dice.

#### (a) Dice roll code:

In [1]:
import random

def dice_roll(n):
    if n < 1:
        raise ValueError("A die must have at least 1 side.")
    return random.randint(1, n)

## tests ##
for n in range(1, 20):
    for j in range(100):
        value = dice_roll(n) 
        assert value >= 1 and value <= n
        
dice_roll(80)

# fairness test #

def test_dice_fair(num_rolls: int = 100000):
    counts = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0}
    
    for _ in range(num_rolls):
        result = dice_roll(6)
        counts[result] += 1
        
   
    print(f"Results after {num_rolls:,} rolls") 
    print(f"Target ideal fraction (1/6): ~0.1667\n")
    
    for side, tally in counts.items():
        observed_fraction = tally / num_rolls
        difference = abs(observed_fraction - 1/6)
        print(f"Side {side}: Count = {tally:<6} | Observed Fraction = {observed_fraction:.4f}")
        
        
test_dice_fair()

Results after 100,000 rolls
Target ideal fraction (1/6): ~0.1667

Side 1: Count = 16718  | Observed Fraction = 0.1672
Side 2: Count = 16505  | Observed Fraction = 0.1651
Side 3: Count = 16600  | Observed Fraction = 0.1660
Side 4: Count = 16737  | Observed Fraction = 0.1674
Side 5: Count = 16736  | Observed Fraction = 0.1674
Side 6: Count = 16704  | Observed Fraction = 0.1670


In [ ]:
## tests ##
for n in range(1, 20):
    for j in range(100):
        value = dice_roll(n) 
        assert value >= 1 and value <= n

#### (b) Test for fairness

## Exercise 05.2 (variance estimation)

For a random variable $X$, the variance of $X$ is defined as  

$$
\begin{align}
\mathrm{Var}\left[ X \right] &= \mathrm{E}\left[\left( X - \mu \right)^2\right]  \\ 
&= \mathrm{E}\left[ X^{2} \right] - \mathrm{E}\left[ X \right]^2
\end{align}
$$

where $\mathrm{E}$ is the 'expectation' (mean of something) and $\mu = \mathrm{E}(X)$ is the mean of $X$. If we have all data (the entire 'population'),
the variance can be computed from:

$$
\mathrm{Var}\left[ X \right] = \frac{\sum_{i=0}^{N-1} x^{2}_{i}}{N} - \left( \frac{ \left( \sum_{i=0}^{N-1} x_{i} \right)}{N} \right)^{2}
$$

Often, we only have a sample of data. For example, we might want to estimate the variance in height for students at a university using just a random sample of students. 
When using a sample from a larger data set to estimate the variance, the above formula has a *bias*. Therefore, it is common to use the *unbiased* estimator

$$
s^{2} = \left( \frac{\sum_{i=0}^{n-1} x^{2}_{i}}{n} - \left( \frac{ \left( \sum_{i=0}^{n-1} x_{i} \right)}{n} \right)^{2} \right) \frac{n}{n-1}
$$

to estimate the variance. In this exercise we will use the unbiased estimator.

1. Create a function that returns the estimated variance for a list of numbers based on the above equation. Test your function using 1 million samples drawn from a Gaussian distribution with a mean of 10 and a standard deviation of 3. Use your crsid to seed the random number generator
   
   *See hint below on how to create the sample.*

2. For a sample drawn from a distribution with mean $5 \times 10^6$ and standard deviation $2.0$, estimate the variance using (i) your function for estimating the variance, and (ii) using the `variance` function from the Python `statistics` module. Comment on and explain any significant differences in the results from (i) and (ii).

### Hint: sampling from a distribution

The function `random.gauss` can be used to sample a Gaussian distribution with a specified mean and standard deviation (square root of the variance) *N* times, e.g.:

In [ ]:
import random

random.seed("gnw20")  # See the random number generator

mu = 10.0    # mean of the distribution
sigma = 3.0  # Standard deviation
x = [random.gauss(mu, sigma) for i in range(8)]
print(x)

### Solution

1. Estimate variance

In [ ]:
import math
import random

# mean, sd and number of samples
random.seed("gnw20")  
mu = 10.0      
sigma = 3.0   
N = 1000000   


x = [random.gauss(mu, sigma) for i in range(N)]

def Var(data):
    n = len(data)
    if n < 2:
        raise ValueError("Variance requires at least two data points.")
        
    sum_x = sum(data)
    sum_x_sq = sum(val**2 for val in data)
    
    # Algebraically equivalent to: (1 / (n - 1)) * sum((x - mean)^2)
    estimated_variance = (n / (n - 1)) * (((sum_x_sq)/n) - ((sum_x)/n)**2)
    
    return estimated_variance  

variance_custom = Var(x)
print(f"(i) Custom Variance Function:     {variance_custom:.12f}")

2. Use the `statistics` module to estimate the variance, and compare the estimated variance using `statistics` to the estimated variance using your implementation for the variance estimation.

In [3]:
import statistics
variance_custom = Var(x)
variance_stats = statistics.variance(x)

print(f"(i) Custom Variance Function:     {variance_custom:.12f}")
print(f"(ii) Statistics Module Variance:  {variance_stats:.12f}")
# test
assert math.isclose(variance_custom, variance_stats, rel_tol=1e-12)

(i) Custom Variance Function:     9.002113238849
(ii) Statistics Module Variance:  9.002113238849


## Exercise 05.03 (optional, parallel processing)

Almost all modern computer processing units have multiple processing *cores*. To utilise the full performance of a processor, operations need to be performed in *parallel*, i.e. each processing core is given a task to perform.

Parallel computing is a very rich and technical area. To help exploit multi-core systems there are libraries that support parallel processing. Below is a simple example using the Python `multiprocessing` library.

In [ ]:
import multiprocessing

# On some operaring systems, you may need to uncomment the below line
# multiprocessing.set_start_method('fork')

def f(task):
    """A function that print the input argument and the id for the process that executes the function"""
    print(f"Task index {task}, process id: {multiprocessing.current_process()}\n")
     
    return str(multiprocessing.current_process())
        
        
# Using 3 'processes', execute the function 'f' four times (each time with a different argument)  
with multiprocessing.Pool(processes=3) as p:
    procs = p.map(f, [0, 1, 2, 3])  # Call function mysort three times
    
    print("Returned data (a list)") 
    print(procs)

Process SpawnPoolWorker-1:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/queues.py", line 387, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
AttributeError: module '__main__' has no attribute 'f'
Process SpawnPoolWorker-2:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/process.py", line 320, in _boo

Consider the below code that creates a list of lists of sorted numbers.

In [2]:
import random

In [1]:
def mysort(N):
    """Create a randomly ordered list of integers of length N, and return the sorted list"""
    # Create randomly ordered list
    x = random.sample(range(0, N), N)
 
    # Return sorted list of numbers
    return sorted(x)

def sorted_lists(N, p):
    """Create a list of sorted lists"""
    data = []
    for i in range(p):
        data.append(mysort(N))

%time x = sorted_lists(1000000, 5)

CPU times: user 14 μs, sys: 3 μs, total: 17 μs
Wall time: 21.2 μs


NameError: name 'random' is not defined

Use the `multiprocessing` module to perform the above operation in parallel. Investigate how the processing time changes with the number of processes, and in particular the average time per 'create and sort operation' when changing the number of processes.

In [ ]:
!pip install multiprocess

import random
import time
import multiprocessing
def mysort(N):
    x = random.sample(range(0, N), N)
    return sorted(x)
    
N = 100000
p = 5
def parallel_sorted_lists(N, p, num_processes):
    with multiprocessing.Pool(num_processes) as pool:
        data = pool.map(mysort, [N] * p)
    return data

if __name__ == '__main__':
    print(f"{'Processes':<12}{'Total Time (s)':<18}{'Avg Time per Task (s)'}")
    print("-" * 55)
    
    for num_processes in [1,2,4,5]:
        start_time = time.time()
        _ = parallel_sorted_lists(N, p, num_processes)
        total_time = time.time() - start_time
        avg_time_per_task = total_time / p 

print(f"{num_processes:<12}{total_time:<18.4f}{avg_time_per_task:.4f}")

Processes   Total Time (s)    Avg Time per Task (s)
-------------------------------------------------------


Process SpawnPoolWorker-1:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/queues.py", line 387, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
AttributeError: module '__main__' has no attribute 'mysort'
Process SpawnPoolWorker-2:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/process.py", line 320, in

In [1]:
import multiprocessing

# On some operaring systems, you may need to uncomment the below line
multiprocessing.set_start_method('fork')

def f(task):
    """A function that print the input argument and the id for the process that executes the function"""
    print(f"Task index {task}, process id: {multiprocessing.current_process()}\n")
     
    return str(multiprocessing.current_process())
        
        
# Using 3 'processes', execute the function 'f' four times (each time with a different argument)  
with multiprocessing.Pool(processes=3) as p:
    procs = p.map(f, [0, 1, 2, 3])  # Call function mysort three times
    
    print("Returned data (a list)") 
    print(procs)

Task index 0, process id: <ForkProcess name='ForkPoolWorker-1' parent=67710 started daemon>
Task index 1, process id: <ForkProcess name='ForkPoolWorker-2' parent=67710 started daemon>
Task index 2, process id: <ForkProcess name='ForkPoolWorker-3' parent=67710 started daemon>



Task index 3, process id: <ForkProcess name='ForkPoolWorker-2' parent=67710 started daemon>

Returned data (a list)
["<ForkProcess name='ForkPoolWorker-1' parent=67710 started daemon>", "<ForkProcess name='ForkPoolWorker-2' parent=67710 started daemon>", "<ForkProcess name='ForkPoolWorker-3' parent=67710 started daemon>", "<ForkProcess name='ForkPoolWorker-2' parent=67710 started daemon>"]
